In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pytz
import yfinance as yf
import pyodbc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import time
import logging
import pandas as pd
from truedata import TD_hist

In [1]:
def fetch_truedata_history(
    ticker_list: list,
    duration: str = '1 Y',
    bar_size: str = 'EOD',
    sleep_time: float = 0.1
) -> tuple[pd.DataFrame, list]:
    """
    Fetches historical data from TrueData for a list of tickers.

    Parameters
    ----------
    username : str
        TrueData username.
    password : str
        TrueData password.
    ticker_list : list
        List of ticker symbols to fetch data for.
    duration : str, optional
        Duration of data (e.g., '1 Y', '25 Y', etc.). Default is '1 Y'.
    bar_size : str, optional
        Bar size for data ('EOD', 'WEEK', etc.). Default is 'EOD'.
    sleep_time : float, optional
        Delay between API calls to avoid throttling. Default is 0.2 seconds.

    Returns
    -------
    final_df : pd.DataFrame
        Combined DataFrame of all tickers' historical data.
    error_list : list
        List of tickers that failed to fetch.
    """
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    username =  'tdwsf695'
    password = 'ocean@695'
    # Initialize connection
    td_hist = TD_hist(username, password)
    df_list = []
    error_list = []
    for ticker in ticker_list:
        try:
            df = td_hist.get_historic_data([ticker], duration=duration, bar_size=bar_size)

            df['Ticker'] = ticker
            df = df.rename(columns={
                'timestamp': 'Date',
                'high': 'High',
                'low': 'Low',
                'close': 'Close',
                'open': 'Open'
            })

            df_list.append(df)
            logging.info(f"Fetched data for {ticker} ({len(df)} rows).")
            time.sleep(sleep_time)

        except Exception as e:
            logging.error(f"Failed to fetch data for {ticker}: {e}")
            error_list.append(ticker)

    final_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()
    return final_df, error_list


In [3]:
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

def process_portfolio(nav_df, ticker_data, initial_value=75, output_file=None):
    """
    Process portfolio allocation and returns final dataframe with portfolio performance.

    Parameters
    ----------
    nav_df : pd.DataFrame
        Dataframe with at least ['Year-Month', 'Ticker'] columns.
    get_individual_stock_data : function
        Function to fetch OHLC data. Must accept (tickers, start_date, end_date) and return DataFrame with ['Date','Ticker','Close'].
    initial_value : float
        Initial portfolio allocation value (default=75).
    debt_ticker : str
        Ticker used as debt/alternative asset (default 'MOGSEC.NS').
    output_file : str or None
        If provided, saves the final dataframe to Excel.

    Returns
    -------
    pd.DataFrame
        Final dataframe with portfolio values.
    """
    df_lis = []
    last_month_value = {}
    last_month_quantity = {}

    year_months = nav_df['Year-Month'].unique()
    for i, year_month in enumerate(year_months):
        print(f"\nProcessing: {year_month}")
        print("Last Month Value:", last_month_value)

        tickers = nav_df[nav_df['Year-Month'] == year_month]['Ticker'].unique()
        year_month_date = pd.to_datetime(f"{year_month}-01")


        prev_month_start = (year_month_date - relativedelta(months=2)).strftime('%Y-%m-%d')
        curr_month_start = year_month_date.strftime('%Y-%m-%d')
        curr_month_end = (year_month_date + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

        # --- Fetch stock data ---
        # stock_data = get_individual_stock_data(tickers, prev_month_start, curr_month_end)
        stock_data = (
            ticker_data[(ticker_data['Date'] >= prev_month_start)
            & (ticker_data['Date'] <= curr_month_end) 
            & (ticker_data['Ticker'].isin(tickers))])
        stock_data = stock_data[stock_data['Date']>='2025-11-11']
        
        # % change
        stock_data['%change'] = stock_data.groupby('Ticker')['Close'].pct_change()
        # Filter current month
        stock_data_flt = stock_data[
            (stock_data['Date'] >= curr_month_start) & (stock_data['Date'] <= curr_month_end)
        ].copy()

        print(stock_data_flt)

        # --- Portfolio allocation logic ---
        if len(last_month_value) == 0:
            # First month → allocate initial portfolio equally
            allocation_per_stock = initial_value / len(tickers)
            stock_allocations = {t: allocation_per_stock for t in tickers}
        else:
            # Continue portfolio
            stock_allocations = {t: last_month_value[t] for t in tickers if t in last_month_value}

            # Pool value of dropped stocks
            dropped_stocks = [t for t in last_month_value if t not in tickers]
            dropped_value = sum(last_month_value[t] for t in dropped_stocks)

            # New stocks → share the dropped value equally
            new_stocks = [t for t in tickers if t not in last_month_value]
            if new_stocks:
                allocation_per_stock = dropped_value / len(new_stocks)
                for t in new_stocks:
                    stock_allocations[t] = allocation_per_stock

        # Apply allocations into dataframe
        for tkr, init_value in stock_allocations.items():
            tkr_idx = stock_data_flt[stock_data_flt['Ticker'] == tkr].index
            stock_data_flt.loc[tkr_idx, 'Buy_Hold_Value'] = init_value * (
                (1 + stock_data_flt.loc[tkr_idx, '%change'].fillna(0)).cumprod()
            )
            
            tkr_df = stock_data_flt[stock_data_flt['Ticker'] == tkr]
            if tkr_df.empty:
                continue

            buy_price = tkr_df.iloc[0]['Close']

            # Carry quantity if stock existed last month
            if tkr in last_month_quantity:
                quantity = last_month_quantity[tkr]
            else:
                quantity = init_value / buy_price
                
            stock_data_flt.loc[tkr_df.index, 'Buy_Price'] = buy_price
            stock_data_flt.loc[tkr_df.index, 'Quantity'] = quantity

        # --- Update last month values ---

        last_month_quantity = (
            stock_data_flt.groupby('Ticker')['Quantity'].last().to_dict()
        )
        
        last_month_value = (
            stock_data_flt.groupby('Ticker')['Buy_Hold_Value'].last().to_dict()
        )

        # --- Track total portfolio value ---
        stock_data_flt['Total_Portfolio_Value'] = (
            stock_data_flt.groupby('Date')['Buy_Hold_Value'].transform('sum')
        )

        df_lis.append(stock_data_flt)

    
    final_df = pd.concat(df_lis).reset_index(drop=True)

    # if output_file:
    #     final_df.to_excel(output_file, index=False)

    return final_df



In [4]:
import os
import pandas as pd

def prepare_and_process_portfolio(input_file, start_date, end_date, output_folder,
                                  process_portfolio,
                                  equity_allocation=75, gold_allocation=25):
    """
    Prepare portfolio dataframe with momentum stocks + GOLDBEES and process performance.

    Parameters
    ----------
    input_file : str
        Path to momentum Excel file (with End_Date, Ticker columns).
    start_date : str (YYYY-MM-DD)
        Start date for filtering.
    end_date : str (YYYY-MM-DD)
        End date for filtering.
    output_folder : str
        Folder to save output file.
    get_individual_stock_data : function
        Function to fetch stock NAV/price data.
    process_pocrtfolio : function
        Function to process equity portion of portfolio.
    process_gold : function
        Function to process gold portion of portfolio.
    equity_allocation : int, optional
        Initial allocation to equities (default=75000).
    gold_allocation : int, optional
        Initial allocation to gold (default=25000).

    Returns
    -------
    final_df : pd.DataFrame
        Combined portfolio dataframe.
    """

    # Load and clean
    nav_df = pd.read_excel(input_file).rename(columns={'End_Date': 'Date'})
    nav_df['Date'] = pd.to_datetime(nav_df['Date'])
    nav_df = (
        nav_df[(nav_df['Date'] >= start_date) & (nav_df['Date'] <= end_date)]
        .reset_index(drop=True)[['Date', 'Ticker']]
    )
    nav_df['Year-Month'] = nav_df['Date'].dt.to_period('M').astype(str)
    stocks = pd.read_excel(input_file)

    # Add GOLDBEES for each unique date
    goldbees_df = pd.DataFrame({
        'Date': nav_df['Date'].unique(),
        'Ticker': 'GOLDBEES'
    })
    goldbees_df['Year-Month'] = pd.to_datetime(goldbees_df['Date']).dt.to_period('M').astype(str)
    # print(goldbees_df)

    # Combine
    concat_df = (
        pd.concat([nav_df, goldbees_df], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )

    # symbol_list = stocks['Ticker'].unique()
    # ticker_data = fetch_truedata_history(
    #     ticker_list = symbol_list,
    #     duration = '5 Y',
    #     bar_size = 'EOD',
    #     sleep_time= 0.1
    # )[0]
    # final_df = process_portfolio(concat_df, ticker_data, equity_allocation)

    
    # Split
    ticker_df = concat_df.query("Ticker != 'GOLDBEES'")
    symbol_list = ticker_df['Ticker'].unique()
    ticker_data_other_stocks = fetch_truedata_history(
        ticker_list = symbol_list,
        duration = '10 Y',
        bar_size = 'EOD',
        sleep_time= 0.1
    )[0]

    
    gold_df = concat_df.query("Ticker == 'GOLDBEES'")
    symbol_list = gold_df['Ticker'].unique()
    ticker_data_gold = fetch_truedata_history(
        ticker_list = symbol_list,
        duration = '10 Y',
        bar_size = 'EOD',
        sleep_time= 0.1
    )[0]
    # print(gold_df)

    # Process
    final_df_other_stocks = process_portfolio(ticker_df, ticker_data_other_stocks, equity_allocation)
    # final_df_gold = process_gold(gold_df, get_individual_stock_data, gold_allocation)
    final_df_gold = process_portfolio(gold_df, ticker_data_gold, gold_allocation)


    # Merge results
    final_df = (
        pd.concat([final_df_other_stocks, final_df_gold], ignore_index=True)
          .sort_values(['Date', 'Ticker'])
          .reset_index(drop=True)
    )


    # --- ensure output folder exists ---
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # --- extract middle folder name from input path ---
    middle_folder = os.path.basename(os.path.dirname(input_file))
    # e.g. for path ".../nifty500_21April2025_results/master_momentum_summary.xlsx"
    # middle_folder = "nifty500_21April2025_results"

    # --- create output filename using middle folder ---
    output_file = os.path.join(output_folder, f"{middle_folder}_gold_buy&hold_returns.xlsx")

    # --- save output ---
    # final_df.to_excel(output_file, index=False)
    print(f"✅ Final output saved to: {output_file}")

    return final_df


In [5]:
#NSE500

In [6]:
final_df = prepare_and_process_portfolio(
    input_file="Stocks/Nifty_500_2025_Apr_20_stocks_results/master_momentum_summary.xlsx",
    start_date="2025-11-01",
    end_date="2026-01-01",
    output_folder="Trials",
    process_portfolio=process_portfolio
)

import plotly.express as px

# ✅ Group by Date and calculate total portfolio value
portfolio_summary = (
    final_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)

# ✅ Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # 🔑 width
                  height=500)    # 🔑 height

fig.show()

(2026-01-31 09:39:50,336) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:12212 Thread:14348)
2026-01-31 09:39:50,336 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-01-31 09:39:50,834 - INFO - Fetched data for AIIL (442 rows).
2026-01-31 09:39:51,462 - INFO - Fetched data for ANANDRATHI (1026 rows).
2026-01-31 09:39:52,141 - INFO - Fetched data for CANBK (2476 rows).
2026-01-31 09:39:52,800 - INFO - Fetched data for CUMMINSIND (2476 rows).
2026-01-31 09:39:53,397 - INFO - Fetched data for DELHIVERY (917 rows).
2026-01-31 09:39:54,071 - INFO - Fetched data for FORTIS (2476 rows).
2026-01-31 09:39:54,695 - INFO - Fetched data for GLAND (1287 rows).
2026-01-31 09:39:55,366 - INFO - Fetched data for HBLENGINE (2476 rows).
2026-01-31 09:39:56,018 - INFO - Fetched data for HEROMOTOCO (2476 rows).
2026-01-31 09:39:56,679 - INFO - Fetched data for HINDALCO (2476 rows).
2026-01-31 09:39:57,507 - INFO - Fetched data for HINDCOPPER (


Processing: 2025-11
Last Month Value: {}
            Date    Open    High     Low    Close   volume  oi   Ticker  \
386   2025-11-11   543.8   556.5   532.0   547.95  1343800   0     AIIL   
387   2025-11-12   544.0   562.4   540.2   559.60   337825   0     AIIL   
388   2025-11-13   560.0   575.8   560.0   570.80   264485   0     AIIL   
389   2025-11-14   564.0   569.0   554.0   554.90   174175   0     AIIL   
390   2025-11-17   550.0   568.0   544.0   556.75   645580   0     AIIL   
...          ...     ...     ...     ...      ...      ...  ..      ...   
36790 2025-11-24  7550.0  7550.0  7390.5  7427.00   305698   0  POLYCAB   
36791 2025-11-25  7420.0  7520.0  7384.0  7439.00   130117   0  POLYCAB   
36792 2025-11-26  7429.5  7568.5  7416.0  7539.00   132131   0  POLYCAB   
36793 2025-11-27  7560.0  7560.0  7425.0  7479.00   170964   0  POLYCAB   
36794 2025-11-28  7489.0  7520.0  7451.5  7470.00   223708   0  POLYCAB   

        %change  
386         NaN  
387    0.021261  
388

In [7]:
final_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value
0,2025-11-11,543.8,556.50,532.00,547.95,1343800,0,AIIL,NaN,3.750000,547.95,0.006844,75.000000
1,2025-11-11,3125.0,3139.90,3040.00,3084.90,151685,0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.000000
2,2025-11-11,140.6,141.39,137.95,140.87,21813910,0,CANBK,NaN,3.750000,140.87,0.026620,75.000000
3,2025-11-11,4349.1,4420.00,4290.90,4414.20,395014,0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.000000
4,2025-11-11,430.1,434.90,424.45,430.10,4395363,0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171,2026-01-30,234.8,238.50,233.05,237.50,8061305,0,NYKAA,-0.001681,3.269445,265.75,0.013891,74.683939
1172,2026-01-30,2700.0,2849.00,2694.90,2828.10,626081,0,RADICO,0.046940,3.184223,3253.90,0.001141,74.683939
1173,2026-01-30,1064.0,1082.50,1060.40,1077.15,9480769,0,SBIN,0.010270,4.072755,984.75,0.003771,74.683939
1174,2026-01-30,1025.0,1029.30,1010.15,1020.00,6464610,0,SHRIRAMFIN,-0.002738,3.802469,1019.70,0.003642,74.683939


In [8]:
final_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value
0,2025-11-11,543.8,556.50,532.00,547.95,1343800,0,AIIL,NaN,3.750000,547.95,0.006844,75.000000
1,2025-11-11,3125.0,3139.90,3040.00,3084.90,151685,0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.000000
2,2025-11-11,140.6,141.39,137.95,140.87,21813910,0,CANBK,NaN,3.750000,140.87,0.026620,75.000000
3,2025-11-11,4349.1,4420.00,4290.90,4414.20,395014,0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.000000
4,2025-11-11,430.1,434.90,424.45,430.10,4395363,0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171,2026-01-30,234.8,238.50,233.05,237.50,8061305,0,NYKAA,-0.001681,3.269445,265.75,0.013891,74.683939
1172,2026-01-30,2700.0,2849.00,2694.90,2828.10,626081,0,RADICO,0.046940,3.184223,3253.90,0.001141,74.683939
1173,2026-01-30,1064.0,1082.50,1060.40,1077.15,9480769,0,SBIN,0.010270,4.072755,984.75,0.003771,74.683939
1174,2026-01-30,1025.0,1029.30,1010.15,1020.00,6464610,0,SHRIRAMFIN,-0.002738,3.802469,1019.70,0.003642,74.683939


In [9]:
old_df = final_df[~((final_df['Date']>'2025-11-30') & (final_df['Ticker']=='GOLDBEES'))]
old_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value
0,2025-11-11,543.8,556.50,532.00,547.95,1343800,0,AIIL,NaN,3.750000,547.95,0.006844,75.000000
1,2025-11-11,3125.0,3139.90,3040.00,3084.90,151685,0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.000000
2,2025-11-11,140.6,141.39,137.95,140.87,21813910,0,CANBK,NaN,3.750000,140.87,0.026620,75.000000
3,2025-11-11,4349.1,4420.00,4290.90,4414.20,395014,0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.000000
4,2025-11-11,430.1,434.90,424.45,430.10,4395363,0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171,2026-01-30,234.8,238.50,233.05,237.50,8061305,0,NYKAA,-0.001681,3.269445,265.75,0.013891,74.683939
1172,2026-01-30,2700.0,2849.00,2694.90,2828.10,626081,0,RADICO,0.046940,3.184223,3253.90,0.001141,74.683939
1173,2026-01-30,1064.0,1082.50,1060.40,1077.15,9480769,0,SBIN,0.010270,4.072755,984.75,0.003771,74.683939
1174,2026-01-30,1025.0,1029.30,1010.15,1020.00,6464610,0,SHRIRAMFIN,-0.002738,3.802469,1019.70,0.003642,74.683939


In [10]:
old_df['Ticker'].unique()

array(['AIIL', 'ANANDRATHI', 'CANBK', 'CUMMINSIND', 'DELHIVERY', 'FORTIS',
       'GLAND', 'GOLDBEES', 'HBLENGINE', 'HEROMOTOCO', 'HINDALCO',
       'HINDCOPPER', 'HYUNDAI', 'IIFL', 'INDIANB', 'INTELLECT',
       'LAURUSLABS', 'LTF', 'NETWEB', 'PAYTM', 'POLYCAB', 'ABCAPITAL',
       'AUBANK', 'EICHERMOT', 'LTIM', 'M&MFIN', 'MARUTI', 'MCX',
       'MUTHOOTFIN', 'NATIONALUM', 'NYKAA', 'SYRMA', 'GPIL', 'JKTYRE',
       'JSL', 'RADICO', 'SBIN', 'SHRIRAMFIN', 'VEDL'], dtype=object)

In [11]:
df = fetch_truedata_history(
    ticker_list = ['GOLDBEES', 'SILVERBEES', 'MOGSEC'],
    duration = '5 Y',
    bar_size = 'EOD',
    sleep_time= 0.1
)[0]
df = df[["Date","Ticker", "Open", "Close"]]
df['%change'] = df['Close'].pct_change()
df = df[df['Date'] >= '2025-12-01']
# df.to_excel('C:\\Users\\Admin\\Momentum\\Automating Momentum True Data\\Trials\\nse200_Nifty_200_2025_Aug_nse200_nse200_nse200_nse200_nse200_returns.xlsx', index=False)
df

(2026-01-31 09:40:18,530) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:12212 Thread:14348)
(2026-01-31 09:40:18,530) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:12212 Thread:14348)
(2026-01-31 09:40:18,530) WARNING :: Connected successfully to TrueData Historical Data Service...  (PID:12212 Thread:14348)
2026-01-31 09:40:18,530 - WARNING - Connected successfully to TrueData Historical Data Service... 
2026-01-31 09:40:19,036 - INFO - Fetched data for GOLDBEES (1240 rows).
2026-01-31 09:40:19,613 - INFO - Fetched data for SILVERBEES (988 rows).
2026-01-31 09:40:20,212 - INFO - Fetched data for MOGSEC (1150 rows).


,Date,Ticker,Open,Close,%change
1198,2025-12-01,GOLDBEES,106.09,106.72,0.020073
1199,2025-12-02,GOLDBEES,106.65,105.63,-0.010214
1200,2025-12-03,GOLDBEES,106.25,106.51,0.008331
1201,2025-12-04,GOLDBEES,106.67,105.92,-0.005539
1202,2025-12-05,GOLDBEES,106.27,106.89,0.009158
...,...,...,...,...,...
3373,2026-01-23,MOGSEC,62.90,62.99,0.000953
3374,2026-01-27,MOGSEC,62.99,62.77,-0.003493
3375,2026-01-28,MOGSEC,62.94,62.99,0.003505
3376,2026-01-29,MOGSEC,62.83,62.90,-0.001429


In [12]:
ticker_weights = {'GOLDBEES':0.6, 'SILVERBEES':0.2, 'MOGSEC':0.2}
# factor = 51214.0199725867
factor = 25.32190919


ticker_value = {ticker: weight * factor for ticker, weight in ticker_weights.items()}
ticker_value

{'GOLDBEES': 15.193145514, 'SILVERBEES': 5.064381838, 'MOGSEC': 5.064381838}

In [13]:
df['BaseValue'] = df['Ticker'].map(ticker_value)
# Assign values
df['Value'] = df['Ticker'].map(ticker_value)
# Convert %change to numeric (if needed)
df['%change'] = pd.to_numeric(df['%change'])
# Sort (important for cumprod)
df = df.sort_values(['Ticker', 'Date'])

# Daily growth factor
df['ret_factor'] = 1 + df['%change']

# Cumulative factor per ticker
df['cum_factor'] = df.groupby('Ticker')['ret_factor'].cumprod()

# FINAL DAILY VALUE
df['Value_On_Date'] = df['BaseValue'] * df['cum_factor']

df = df[['Date', 'Ticker', 'Open', 'Close', 'Value_On_Date', '%change']].rename(columns={'Value_On_Date':'Buy_Hold_Value'}).sort_values(by='Date')
df

,Date,Ticker,Open,Close,Buy_Hold_Value,%change
1198,2025-12-01,GOLDBEES,106.09,106.72,15.498112,0.020073
2186,2025-12-01,SILVERBEES,166.02,166.21,5.369337,0.060216
3336,2025-12-01,MOGSEC,62.70,62.82,5.053121,-0.002224
1199,2025-12-02,GOLDBEES,106.65,105.63,15.339820,-0.010214
2187,2025-12-02,SILVERBEES,167.01,165.34,5.341232,-0.005234
...,...,...,...,...,...,...
1238,2026-01-29,GOLDBEES,143.00,146.53,21.279407,0.078854
3376,2026-01-29,MOGSEC,62.83,62.90,5.059556,-0.001429
3377,2026-01-30,MOGSEC,62.71,63.06,5.072426,0.002544
1239,2026-01-30,GOLDBEES,138.70,131.12,19.041534,-0.105166


In [14]:
# df

In [15]:
df['Buy_Value'] = df.groupby('Ticker')['Buy_Hold_Value'].transform('first')
df['Buy_Price'] = df.groupby('Ticker')['Close'].transform('first')
df['Quantity'] = df['Buy_Value'] / df['Buy_Price']
df

,Date,Ticker,Open,Close,Buy_Hold_Value,%change,Buy_Value,Buy_Price,Quantity
1198,2025-12-01,GOLDBEES,106.09,106.72,15.498112,0.020073,15.498112,106.72,0.145222
2186,2025-12-01,SILVERBEES,166.02,166.21,5.369337,0.060216,5.369337,166.21,0.032305
3336,2025-12-01,MOGSEC,62.70,62.82,5.053121,-0.002224,5.053121,62.82,0.080438
1199,2025-12-02,GOLDBEES,106.65,105.63,15.339820,-0.010214,15.498112,106.72,0.145222
2187,2025-12-02,SILVERBEES,167.01,165.34,5.341232,-0.005234,5.369337,166.21,0.032305
...,...,...,...,...,...,...,...,...,...
1238,2026-01-29,GOLDBEES,143.00,146.53,21.279407,0.078854,15.498112,106.72,0.145222
3376,2026-01-29,MOGSEC,62.83,62.90,5.059556,-0.001429,5.053121,62.82,0.080438
3377,2026-01-30,MOGSEC,62.71,63.06,5.072426,0.002544,5.053121,62.82,0.080438
1239,2026-01-30,GOLDBEES,138.70,131.12,19.041534,-0.105166,15.498112,106.72,0.145222


In [16]:
conc_df = pd.concat([old_df, df])
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value,Buy_Value
0,2025-11-11,543.80,556.50,532.00,547.95,1343800.0,0.0,AIIL,NaN,3.750000,547.95,0.006844,75.0,NaN
1,2025-11-11,3125.00,3139.90,3040.00,3084.90,151685.0,0.0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.0,NaN
2,2025-11-11,140.60,141.39,137.95,140.87,21813910.0,0.0,CANBK,NaN,3.750000,140.87,0.026620,75.0,NaN
3,2025-11-11,4349.10,4420.00,4290.90,4414.20,395014.0,0.0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.0,NaN
4,2025-11-11,430.10,434.90,424.45,430.10,4395363.0,0.0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1238,2026-01-29,143.00,NaN,NaN,146.53,NaN,NaN,GOLDBEES,0.078854,21.279407,106.72,0.145222,NaN,15.498112
3376,2026-01-29,62.83,NaN,NaN,62.90,NaN,NaN,MOGSEC,-0.001429,5.059556,62.82,0.080438,NaN,5.053121
3377,2026-01-30,62.71,NaN,NaN,63.06,NaN,NaN,MOGSEC,0.002544,5.072426,62.82,0.080438,NaN,5.053121
1239,2026-01-30,138.70,NaN,NaN,131.12,NaN,NaN,GOLDBEES,-0.105166,19.041534,106.72,0.145222,NaN,15.498112


In [17]:
import plotly.express as px

# ✅ Group by Date and calculate total portfolio value
portfolio_summary = (
    conc_df.groupby("Date", as_index=False)["Buy_Hold_Value"].sum()
)
# ✅ Plot with Plotly
fig = px.line(
    portfolio_summary,
    x="Date",
    y="Buy_Hold_Value",
    title="Buy_Hold_Value Over Time",
    labels={"Date": "Date", "Buy_Hold_Value": "Buy_Hold_Value"},
    markers=True
)

fig.update_traces(line=dict(width=2))
fig.update_layout(width=1000,   # 🔑 width
                  height=500)    # 🔑 height
fig.show()

In [18]:
# Momentum/Automating Momentum True Data/Trials/Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx

In [19]:
 # conc_df.to_excel('C://Users//Admin//Momentum//Automating Momentum True Data//Trials//Nifty_500_2025_Apr_20_stocks_results_GoldSilverDebt_buy&hold_returns.xlsx', index=False)

In [20]:
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value,Buy_Value
0,2025-11-11,543.80,556.50,532.00,547.95,1343800.0,0.0,AIIL,NaN,3.750000,547.95,0.006844,75.0,NaN
1,2025-11-11,3125.00,3139.90,3040.00,3084.90,151685.0,0.0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.0,NaN
2,2025-11-11,140.60,141.39,137.95,140.87,21813910.0,0.0,CANBK,NaN,3.750000,140.87,0.026620,75.0,NaN
3,2025-11-11,4349.10,4420.00,4290.90,4414.20,395014.0,0.0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.0,NaN
4,2025-11-11,430.10,434.90,424.45,430.10,4395363.0,0.0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1238,2026-01-29,143.00,NaN,NaN,146.53,NaN,NaN,GOLDBEES,0.078854,21.279407,106.72,0.145222,NaN,15.498112
3376,2026-01-29,62.83,NaN,NaN,62.90,NaN,NaN,MOGSEC,-0.001429,5.059556,62.82,0.080438,NaN,5.053121
3377,2026-01-30,62.71,NaN,NaN,63.06,NaN,NaN,MOGSEC,0.002544,5.072426,62.82,0.080438,NaN,5.053121
1239,2026-01-30,138.70,NaN,NaN,131.12,NaN,NaN,GOLDBEES,-0.105166,19.041534,106.72,0.145222,NaN,15.498112


In [21]:
conc_df['Asset Type'] = np.where(
    conc_df['Ticker'] == 'GOLDBEES', 'Gold',
    np.where(
        conc_df['Ticker'] == 'SILVERBEES', 'Silver',
        np.where(
            conc_df['Ticker'] == 'MOGSEC', 'Debt',
            'Equities'
        )
    )
)

conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value,Buy_Value,Asset Type
0,2025-11-11,543.80,556.50,532.00,547.95,1343800.0,0.0,AIIL,NaN,3.750000,547.95,0.006844,75.0,NaN,Equities
1,2025-11-11,3125.00,3139.90,3040.00,3084.90,151685.0,0.0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.0,NaN,Equities
2,2025-11-11,140.60,141.39,137.95,140.87,21813910.0,0.0,CANBK,NaN,3.750000,140.87,0.026620,75.0,NaN,Equities
3,2025-11-11,4349.10,4420.00,4290.90,4414.20,395014.0,0.0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.0,NaN,Equities
4,2025-11-11,430.10,434.90,424.45,430.10,4395363.0,0.0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.0,NaN,Equities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1238,2026-01-29,143.00,NaN,NaN,146.53,NaN,NaN,GOLDBEES,0.078854,21.279407,106.72,0.145222,NaN,15.498112,Gold
3376,2026-01-29,62.83,NaN,NaN,62.90,NaN,NaN,MOGSEC,-0.001429,5.059556,62.82,0.080438,NaN,5.053121,Debt
3377,2026-01-30,62.71,NaN,NaN,63.06,NaN,NaN,MOGSEC,0.002544,5.072426,62.82,0.080438,NaN,5.053121,Debt
1239,2026-01-30,138.70,NaN,NaN,131.12,NaN,NaN,GOLDBEES,-0.105166,19.041534,106.72,0.145222,NaN,15.498112,Gold


In [22]:
# conc_df = conc_df[conc_df['Date']<='2026-01-06']
conc_df = conc_df[conc_df['Date'] <= pd.Timestamp.today().normalize()]
conc_df

,Date,Open,High,Low,Close,volume,oi,Ticker,%change,Buy_Hold_Value,Buy_Price,Quantity,Total_Portfolio_Value,Buy_Value,Asset Type
0,2025-11-11,543.80,556.50,532.00,547.95,1343800.0,0.0,AIIL,NaN,3.750000,547.95,0.006844,75.0,NaN,Equities
1,2025-11-11,3125.00,3139.90,3040.00,3084.90,151685.0,0.0,ANANDRATHI,NaN,3.750000,3084.90,0.001216,75.0,NaN,Equities
2,2025-11-11,140.60,141.39,137.95,140.87,21813910.0,0.0,CANBK,NaN,3.750000,140.87,0.026620,75.0,NaN,Equities
3,2025-11-11,4349.10,4420.00,4290.90,4414.20,395014.0,0.0,CUMMINSIND,NaN,3.750000,4414.20,0.000850,75.0,NaN,Equities
4,2025-11-11,430.10,434.90,424.45,430.10,4395363.0,0.0,DELHIVERY,NaN,3.750000,430.10,0.008719,75.0,NaN,Equities
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1238,2026-01-29,143.00,NaN,NaN,146.53,NaN,NaN,GOLDBEES,0.078854,21.279407,106.72,0.145222,NaN,15.498112,Gold
3376,2026-01-29,62.83,NaN,NaN,62.90,NaN,NaN,MOGSEC,-0.001429,5.059556,62.82,0.080438,NaN,5.053121,Debt
3377,2026-01-30,62.71,NaN,NaN,63.06,NaN,NaN,MOGSEC,0.002544,5.072426,62.82,0.080438,NaN,5.053121,Debt
1239,2026-01-30,138.70,NaN,NaN,131.12,NaN,NaN,GOLDBEES,-0.105166,19.041534,106.72,0.145222,NaN,15.498112,Gold


In [23]:
conc_df.to_excel('Momentum_Maxfolio.xlsx', index=False)